In [3]:
import json
import random


# ============================================================
# FILE PATHS
# ============================================================

GROUND_TRUTH_FILE = "../community_ground_truth_counts.json"

LLM_FILE = "/home/siu856622573/Code/RQ_4_5/llama/community/llama_community_gt_36.json"

OUTPUT_FILE = "/home/siu856622573/Code/RQ5/llama_community_gt_5adhd_5control_10com.json"


# ============================================================
# LOAD FILES
# ============================================================

with open(GROUND_TRUTH_FILE, "r") as f:
    ground_truth_data = json.load(f)

with open(LLM_FILE, "r") as f:
    llama_data = json.load(f)


# ============================================================
# FIND COMMON SUBJECTS
# ============================================================

common_ids = set(ground_truth_data.keys()) & set(llama_data.keys())

print("Common subjects:", len(common_ids))


# ============================================================
# SEPARATE ADHD AND CONTROL
# ============================================================

adhd_ids = [
    sid for sid in common_ids
    if ground_truth_data[sid]["ground_truth"] == "ADHD"
]

control_ids = [
    sid for sid in common_ids
    if ground_truth_data[sid]["ground_truth"] == "Control"
]


print("Available ADHD:", len(adhd_ids))
print("Available Control:", len(control_ids))


# ============================================================
# CHECK THAT WE HAVE AT LEAST 5 EACH
# ============================================================

if len(adhd_ids) < 5:
    raise ValueError("Less than 5 ADHD subjects available.")

if len(control_ids) < 5:
    raise ValueError("Less than 5 Control subjects available.")


# ============================================================
# SELECT 5 ADHD + 5 CONTROL
# ============================================================

# Random selection
random.seed(42)

selected_adhd = random.sample(adhd_ids, 5)
selected_control = random.sample(control_ids, 5)

selected_ids = selected_adhd + selected_control


print("\nSelected ADHD subjects:")
print(selected_adhd)

print("\nSelected Control subjects:")
print(selected_control)


# ============================================================
# CREATE MERGED OUTPUT
# ============================================================

output = {}


for sid in selected_ids:

    gt_record = ground_truth_data[sid]
    llama_record = llama_data[sid]

    output[sid] = {

        "subject_id": sid,

        # Ground truth from ground-truth file
        "ground_truth": gt_record["ground_truth"],

        # LLM prediction
        "prediction": llama_record.get("prediction"),

        # Actual numerical network values
        "within_network": gt_record.get(
            "within_network",
            {}
        ),

        "between_network": gt_record.get(
            "between_network",
            {}
        ),

        # LLM reasoning
        "reasoning": llama_record.get(
            "reasoning"
        )
    }


# ============================================================
# SAVE JSON
# ============================================================

with open(OUTPUT_FILE, "w") as f:
    json.dump(output, f, indent=2)


print("\nSaved merged file to:")
print(OUTPUT_FILE)


# ============================================================
# SUMMARY
# ============================================================

print("\nFinal dataset:")
print("ADHD:", sum(
    1 for r in output.values()
    if r["ground_truth"] == "ADHD"
))

print("Control:", sum(
    1 for r in output.values()
    if r["ground_truth"] == "Control"
))

Common subjects: 768
Available ADHD: 280
Available Control: 488

Selected ADHD subjects:
['1743472', '2884672', '3856956', '2524687', '9326955']

Selected Control subjects:
['15013', '1139030', '1266183', '2833684', '2659769']

Saved merged file to:
/home/siu856622573/Code/RQ5/llama_community_gt_5adhd_5control_10com.json

Final dataset:
ADHD: 5
Control: 5
